# VISION: CNN-Based Image Classification System

This notebook trains a **Convolutional Neural Network (CNN)** that recognises five classes from the **Fruits 360** dataset:

**Apple, Avocado, Banana, Eggplant, Tomato**

At the end it saves two files that the website (`app.py`) needs:

| File | What it is |
|---|---|
| `model/vision_model.keras` | the trained CNN |
| `class_names.json` | the class order used by the model |

It also saves `model/model_info.json` (versions and test accuracy) so you can install the same TensorFlow version on your computer.

## How to run this notebook in Google Colab

1. Open [colab.research.google.com](https://colab.research.google.com) and choose **File > Upload notebook**, then select `notebook.ipynb`.
2. Turn on the GPU: **Runtime > Change runtime type > Hardware accelerator: T4 GPU > Save**. (A CPU also works, it is just slower.)
3. Choose **Runtime > Run all**.
4. When the last cell finishes, a file called `vision_outputs.zip` is downloaded to your computer.

**Quick test vs full training.** In the settings cell below, `QUICK_TEST = True` uses few images and 3 epochs, so you can check in a few minutes that everything works. Set `QUICK_TEST = False` (the default) for the real training that you submit. The quick test only proves that the code runs: its accuracy will be low, so never submit a quick-test model.

## 1. Settings

All the important numbers are in one place. The class order is **fixed and alphabetical**. The website reads the same order from `class_names.json`, so the notebook and website always agree.

In [ ]:
# ---- Main switch ----------------------------------------------------------
QUICK_TEST = False   # True = fast check (few images, 3 epochs). False = full training.

# ---- Classes (fixed order, also saved to class_names.json) ----------------
CLASS_NAMES = ["Apple", "Avocado", "Banana", "Eggplant", "Tomato"]

# ---- Image and training settings -------------------------------------------
IMAGE_SIZE = 100            # Fruits 360 images are 100 x 100 pixels
BATCH_SIZE = 32
SEED = 42                   # makes sampling and splitting repeatable
CONFIDENCE_THRESHOLD = 0.60 # below this, the website answers "unknown" (saved in model_info.json)

if QUICK_TEST:
    MAX_TRAIN_PER_CLASS = 150    # images per class for training
    MAX_HOLDOUT_PER_CLASS = 60   # images per class for validation + test
    EPOCHS = 3
else:
    MAX_TRAIN_PER_CLASS = 700    # Eggplant has about 700 training images, so 700 keeps classes balanced
    MAX_HOLDOUT_PER_CLASS = 400  # split 50/50 into validation and test
    EPOCHS = 25                  # early stopping usually ends training sooner

# ---- Where the dataset comes from ------------------------------------------
# "github" = official Fruits 360 GitHub repository, no account or key needed (recommended)
# "kaggle" = Kaggle dataset moltean/fruits, needs a kaggle.json API key (see section 2)
DATASET_SOURCE = "github"

print("Quick test mode:" if QUICK_TEST else "Full training mode:",
      f"{MAX_TRAIN_PER_CLASS} train images/class, {MAX_HOLDOUT_PER_CLASS} holdout images/class, {EPOCHS} epochs")

## 2. Import libraries and check the environment

In [ ]:
import os, sys, json, random, shutil, subprocess, time, zipfile
import hashlib, inspect, io, tokenize
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

IN_COLAB = "google.colab" in sys.modules

# Reproducible settings: same random numbers every run (GPU training can still vary slightly).
keras.utils.set_random_seed(SEED)

print("Python     :", sys.version.split()[0])
print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
print("Running in Colab:", IN_COLAB)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "none (training will use the CPU and be slower)")

## 3. Download the Fruits 360 dataset

**Recommended method (no account needed):** the dataset author publishes Fruits 360 on GitHub at `github.com/fruits-360/fruits-360-100x100`. The full repository is large, so the next cell uses a *sparse checkout*: it downloads only the Apple, Avocado, Banana, Eggplant and Tomato folders (about 320 MB) from both the `Training` and `Test` parts.

**Alternative: Kaggle.** Set `DATASET_SOURCE = "kaggle"` in the settings cell. Then:
1. On kaggle.com go to **Settings > API > Create New Token**. This downloads `kaggle.json`.
2. When this cell runs in Colab it asks you to upload `kaggle.json`.
3. Never upload `kaggle.json` to GitHub (it is your password for Kaggle).

The Kaggle download contains the whole dataset (several GB) and its folder layout can change between versions, so the code searches for the folder that contains `Training` and `Test`.

In [ ]:
DATA_DIR = Path("fruits-360")
GITHUB_URL = "https://github.com/fruits-360/fruits-360-100x100.git"

def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

def download_from_github():
    if (DATA_DIR / "Training").exists() and (DATA_DIR / "Test").exists():
        print("Dataset already downloaded, skipping.")
        return
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    run(["git", "clone", "--filter=blob:none", "--no-checkout", "--depth", "1", GITHUB_URL, str(DATA_DIR)])
    # "Apple Red 1/", "Banana 1/", ... (and a plain "Banana/" in case a version has no suffix)
    patterns = [f"/{split}/{name}{suffix}/" for split in ("Training", "Test")
                for name in CLASS_NAMES for suffix in (" *", "")]
    run(["git", "sparse-checkout", "set", "--no-cone", *patterns], cwd=DATA_DIR)
    run(["git", "checkout"], cwd=DATA_DIR)

def download_from_kaggle():
    kaggle_dir = Path.home() / ".kaggle"
    if not (kaggle_dir / "kaggle.json").exists():
        if IN_COLAB:
            from google.colab import files
            print("Upload your kaggle.json file:")
            uploaded = files.upload()
            kaggle_dir.mkdir(exist_ok=True)
            (kaggle_dir / "kaggle.json").write_bytes(uploaded["kaggle.json"])
            os.chmod(kaggle_dir / "kaggle.json", 0o600)
        else:
            raise FileNotFoundError("Put kaggle.json in ~/.kaggle/ first.")
    run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    run(["kaggle", "datasets", "download", "-d", "moltean/fruits", "-p", "kaggle_data", "--unzip"])

def find_dataset_root(base):
    # Return the first folder below `base` that has both Training and Test inside.
    for folder in [Path(base), *sorted(Path(base).rglob("*"))]:
        if folder.is_dir() and (folder / "Training").is_dir() and (folder / "Test").is_dir():
            return folder
    raise FileNotFoundError(f"No folder with Training/ and Test/ found inside {base}")

start = time.time()
if DATASET_SOURCE == "github":
    download_from_github()
    DATASET_ROOT = find_dataset_root(DATA_DIR)
elif DATASET_SOURCE == "kaggle":
    download_from_kaggle()
    DATASET_ROOT = find_dataset_root("kaggle_data")
else:
    raise ValueError("DATASET_SOURCE must be 'github' or 'kaggle'")

TRAIN_DIR = DATASET_ROOT / "Training"
TEST_DIR = DATASET_ROOT / "Test"
print(f"Dataset ready in {time.time() - start:.0f} s at: {DATASET_ROOT}")

## 4. Inspect the dataset folders and verify the five classes

Fruits 360 does **not** have one folder called `Apple`. It has one folder per *variety*, for example `Apple Golden 1`, `Apple Red 2`, `Banana Lady Finger 1`, `Tomato Cherry Red 1`.

**Mapping rule:** a folder belongs to a class when its **first word** is exactly the class name. So `Apple Red 1` becomes *Apple*, but `Pineapple 1` does **not** (its first word is `Pineapple`). All varieties of a class are merged into that class.

In [ ]:
def map_folders(split_dir):
    # Return {class_name: [variety folders]} using the first-word rule.
    mapping = {name: [] for name in CLASS_NAMES}
    for folder in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        first_word = folder.name.split(" ")[0]
        if first_word in mapping:
            mapping[first_word].append(folder)
    return mapping

def list_images(folder):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

all_folders = [p.name for p in TRAIN_DIR.iterdir() if p.is_dir()]
print(f"Folders in Training: {len(all_folders)} (only the downloaded classes are present with the GitHub method)\n")

train_map = map_folders(TRAIN_DIR)
test_map = map_folders(TEST_DIR)

print(f"{'Class':<10}{'Varieties':>10}{'Training imgs':>15}{'Test imgs':>11}")
for name in CLASS_NAMES:
    n_train = sum(len(list_images(f)) for f in train_map[name])
    n_test = sum(len(list_images(f)) for f in test_map[name])
    print(f"{name:<10}{len(train_map[name]):>10}{n_train:>15}{n_test:>11}")
    assert train_map[name] and test_map[name], f"Class {name} was not found in the dataset!"

print("\nAll five classes were found. Example folders:")
for name in CLASS_NAMES:
    print(f"  {name}: {[f.name for f in train_map[name][:4]]}{' ...' if len(train_map[name]) > 4 else ''}")

In [ ]:
# Show one example image from each class
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(13, 3))
for ax, name in zip(axes, CLASS_NAMES):
    folder = train_map[name][0]
    ax.imshow(Image.open(list_images(folder)[0]))
    ax.set_title(f"{name}\n({folder.name})", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Select images and split into training, validation and test sets (no data leakage)

**Why this matters.** Fruits 360 was made by filming each fruit while it rotates. Images that are next to each other in a folder are almost identical. If we split one folder randomly, the validation set would contain near-copies of training images and the accuracy would look better than it really is. This is called **data leakage**.

**How we avoid it:**
* **Training set:** images only from the official `Training` folder.
* **Validation and test sets:** images only from the official `Test` folder, split 50/50 (stratified by class).
* The model never sees any image from the `Test` folder during training.
* A check at the end makes sure no file appears in two sets.

**Balanced sampling:** for each class we take the same maximum number of images, spread evenly across all varieties (round-robin), so Apple (30 varieties) does not drown out Eggplant (2 varieties).

*Note:* validation and test images both come from the official `Test` folder, so they are similar to each other. The test accuracy is still honest with respect to training, but real-world photos (different backgrounds and lighting) are harder than this test set.

In [ ]:
def sample_balanced(folders, max_images, rng):
    # Take up to max_images files, alternating between variety folders.
    per_folder = []
    for folder in folders:
        files = list_images(folder)
        rng.shuffle(files)
        per_folder.append(files)
    chosen, i = [], 0
    while len(chosen) < max_images and any(per_folder):
        files = per_folder[i % len(per_folder)]
        if files:
            chosen.append(files.pop())
        i += 1
    return chosen

rng = random.Random(SEED)
train_paths, train_labels, holdout_paths, holdout_labels = [], [], [], []
for label, name in enumerate(CLASS_NAMES):
    tr = sample_balanced(train_map[name], MAX_TRAIN_PER_CLASS, rng)
    ho = sample_balanced(test_map[name], MAX_HOLDOUT_PER_CLASS, rng)
    train_paths += tr; train_labels += [label] * len(tr)
    holdout_paths += ho; holdout_labels += [label] * len(ho)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    holdout_paths, holdout_labels, test_size=0.5, stratify=holdout_labels, random_state=SEED)

# Leakage checks
assert all(TRAIN_DIR in p.parents for p in train_paths)
assert all(TEST_DIR in p.parents for p in val_paths + test_paths)
assert not set(train_paths) & set(val_paths) and not set(train_paths) & set(test_paths) and not set(val_paths) & set(test_paths)
print("No file is shared between training, validation and test sets.\n")

for split_name, labels in [("Training", train_labels), ("Validation", val_labels), ("Test", test_labels)]:
    counts = np.bincount(labels, minlength=len(CLASS_NAMES))
    print(f"{split_name:<11} {len(labels):>5} images  " + "  ".join(f"{n}={c}" for n, c in zip(CLASS_NAMES, counts)))

## 6. Preprocessing: resize and normalise images consistently

The function below is **copied exactly from `app.py`**. The website uses the same code on uploaded photos, so the model always receives images prepared in the same way:

1. Fix phone photo rotation (EXIF orientation).
2. Put transparent PNG images on a white background.
3. Convert to RGB.
4. Pad to a square with white (Fruits 360 images are already square, so nothing changes for them).
5. Resize to 100 x 100 pixels.

**Normalisation** (dividing pixel values by 255 to get 0-1) happens **inside the model** with a `Rescaling` layer. Because it is part of the saved model, the website cannot forget it.

**Proof that the website uses the same code.** `source_fingerprint()` (also copied from `app.py`) computes a short hash of `preprocess_pil_image()`. The hash is saved in `model_info.json`. When the website starts, it computes the hash of its own copy and refuses to load the model if they differ. The confidence threshold and class order are saved in the same file, and the website reads them from there.

In [ ]:
def preprocess_pil_image(image: Image.Image, image_size: int = 100) -> np.ndarray:
    """Turn any PIL image into a (image_size, image_size, 3) uint8 RGB array.

    Steps:
      1. Rotate the image according to its EXIF orientation (phone photos).
      2. Put transparent images (PNG with alpha) on a white background.
      3. Convert to RGB (3 colour channels).
      4. Pad to a square with white, keeping the whole object visible.
         (Fruits 360 images are square with a white background.)
      5. Resize to image_size x image_size with bilinear interpolation.

    Pixel values stay in the range 0-255. The model divides by 255 itself
    (it has a Rescaling layer), so normalisation can never be forgotten.
    """
    image = ImageOps.exif_transpose(image)

    has_alpha = image.mode in ("RGBA", "LA", "PA") or (
        image.mode == "P" and "transparency" in image.info
    )
    if has_alpha:
        image = image.convert("RGBA")
        white = Image.new("RGBA", image.size, (255, 255, 255, 255))
        image = Image.alpha_composite(white, image)

    image = image.convert("RGB")

    width, height = image.size
    if width != height:
        side = max(width, height)
        square = Image.new("RGB", (side, side), (255, 255, 255))
        square.paste(image, ((side - width) // 2, (side - height) // 2))
        image = square

    if image.size != (image_size, image_size):
        image = image.resize((image_size, image_size), Image.Resampling.BILINEAR)

    return np.asarray(image, dtype=np.uint8)


def source_fingerprint(func) -> str | None:
    """Short hash of a function's code, ignoring comments and blank lines.

    The notebook stores the fingerprint of its preprocess_pil_image() in
    model_info.json. The server compares it with its own copy, so a model
    trained with different preprocessing is refused instead of silently
    giving wrong answers. (Also copied exactly into notebook.ipynb.)
    """
    try:
        source = inspect.getsource(func)
    except (OSError, TypeError):
        return None
    lines = source.splitlines(keepends=True)
    indent = len(lines[0]) - len(lines[0].lstrip())
    source = "".join(line[indent:] if line.strip() else "\n" for line in lines)
    skip = {tokenize.COMMENT, tokenize.NL, tokenize.ENCODING, tokenize.ENDMARKER}
    parts = []
    for tok in tokenize.generate_tokens(io.StringIO(source).readline):
        if tok.type in skip:
            continue
        parts.append(tokenize.tok_name[tok.type] if tok.type in (tokenize.INDENT, tokenize.DEDENT, tokenize.NEWLINE) else tok.string)
    return hashlib.sha256(" ".join(parts).encode("utf-8")).hexdigest()[:16]


def load_images(paths):
    # Load and preprocess a list of image files into one uint8 array.
    images = np.empty((len(paths), IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
    for i, path in enumerate(paths):
        with Image.open(path) as img:
            images[i] = preprocess_pil_image(img, IMAGE_SIZE)
    return images

start = time.time()
x_train, x_val, x_test = load_images(train_paths), load_images(val_paths), load_images(test_paths)
y_train, y_val, y_test = np.array(train_labels), np.array(val_labels), np.array(test_labels)
print(f"Loaded in {time.time() - start:.0f} s")
print("x_train:", x_train.shape, x_train.dtype, "| pixel range:", x_train.min(), "-", x_train.max())
print("x_val  :", x_val.shape)
print("x_test :", x_test.shape)

PREPROCESSING_FINGERPRINT = source_fingerprint(preprocess_pil_image)
print("Preprocessing fingerprint:", PREPROCESSING_FINGERPRINT, "(saved in model_info.json, checked by app.py)")

In [ ]:
# Build fast TensorFlow input pipelines (shuffle training data every epoch)
AUTOTUNE = tf.data.AUTOTUNE
def make_dataset(x, y, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(x_train, y_train, shuffle=True)
val_ds = make_dataset(x_val, y_val, shuffle=False)
test_ds = make_dataset(x_test, y_test, shuffle=False)

## 7. Build the CNN

The model has three parts:

* **Data augmentation** (random flips, rotations, zoom, shifts, contrast). These layers only work during training. They make the model more robust to real photos. During prediction they do nothing.
* **Feature extractor:** four blocks of `Conv2D > BatchNormalization > ReLU > MaxPooling`. Early blocks find edges and colours, later blocks find shapes and textures.
* **Classifier:** `GlobalAveragePooling > Dropout > Dense > Dropout > Dense(5, softmax)`. Softmax turns the output into five probabilities that add up to 1.

Input: a 100 x 100 x 3 image with pixel values 0-255 (the `Rescaling` layer divides by 255).

In [ ]:
def build_model(num_classes=len(CLASS_NAMES)):
    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.1, 0.1, fill_mode="constant", fill_value=255.0),
        layers.RandomContrast(0.2),
    ], name="augmentation")

    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="image")
    x = augmentation(inputs)
    x = layers.Rescaling(1.0 / 255, name="normalize")(x)
    for filters in (32, 64, 128, 128):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization(momentum=0.9)(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="probabilities")(x)
    return keras.Model(inputs, outputs, name="vision_cnn")

model = build_model()
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

## 8. Train the model

* **EarlyStopping** stops training when the validation loss stops improving for 5 epochs and keeps the best weights.
* **ReduceLROnPlateau** lowers the learning rate when progress slows down.

With a T4 GPU, full training usually takes a few minutes. On a CPU it takes longer.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1),
]
start = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=2)
training_minutes = (time.time() - start) / 60
print(f"Training finished in {training_minutes:.1f} minutes ({len(history.history['loss'])} epochs).")

In [ ]:
# Training and validation accuracy / loss graphs
h = history.history
epochs_range = range(1, len(h["loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(epochs_range, h["accuracy"], "o-", label="Training")
ax1.plot(epochs_range, h["val_accuracy"], "o-", label="Validation")
ax1.set_title("Accuracy"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs_range, h["loss"], "o-", label="Training")
ax2.plot(epochs_range, h["val_loss"], "o-", label="Validation")
ax2.set_title("Loss"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Evaluate on the held-out test set

The test set was never used for training or for choosing the best epoch. These are the numbers to report.

In [ ]:
test_probabilities = model.predict(test_ds, verbose=0)
y_pred = test_probabilities.argmax(axis=1)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_accuracy:.4f}  ({test_accuracy * 100:.2f}%)\n")
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

unknown_rate = (test_probabilities.max(axis=1) < CONFIDENCE_THRESHOLD).mean()
print(f"Test images that the website would call 'unknown' (top score < {CONFIDENCE_THRESHOLD:.0%}): {unknown_rate:.1%}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Greens", colorbar=False)
ax.set_title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

## 10. Predict a single image

`predict_image()` does exactly what the website does: preprocess, predict, and answer **unknown** when the best probability is below 60%.

By default it uses an image from the test set. In Colab you can set `UPLOAD_OWN_IMAGE = True` to try your own photo.

In [ ]:
def predict_image(pil_image, model=model):
    pixels = preprocess_pil_image(pil_image, IMAGE_SIZE)
    probabilities = model.predict(pixels[np.newaxis].astype("float32"), verbose=0)[0]
    best = int(probabilities.argmax())
    prediction = CLASS_NAMES[best] if probabilities[best] >= CONFIDENCE_THRESHOLD else "unknown"
    return prediction, probabilities

UPLOAD_OWN_IMAGE = False

if UPLOAD_OWN_IMAGE and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    image = Image.open(name); image.load()
    true_label = "(your image)"
else:
    index = 0
    image = Image.open(test_paths[index]); image.load()
    true_label = CLASS_NAMES[test_labels[index]]

prediction, probabilities = predict_image(image)
print(f"True class : {true_label}")
print(f"Prediction : {prediction}\n")
for name, p in zip(CLASS_NAMES, probabilities):
    print(f"  {name:<9} {p * 100:6.2f}%  {'#' * int(p * 40)}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={"width_ratios": [1, 1.6]})
ax1.imshow(ImageOps.exif_transpose(image)); ax1.axis("off"); ax1.set_title(f"Prediction: {prediction}")
ax2.barh(CLASS_NAMES, probabilities * 100, color=["#2d6a42" if i == probabilities.argmax() else "#b9c7bd" for i in range(len(CLASS_NAMES))])
ax2.axvline(CONFIDENCE_THRESHOLD * 100, color="#c8351f", linestyle="--", label="60% threshold")
ax2.set_xlim(0, 100); ax2.set_xlabel("Confidence (%)"); ax2.invert_yaxis(); ax2.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 11. Save the model and the class mapping

Files are written to the `vision_outputs` folder with the **same layout as the website project**:

```
vision_outputs/
    class_names.json
    model/
        vision_model.keras
        model_info.json
```

In [ ]:
OUTPUT_DIR = Path("vision_outputs")
(OUTPUT_DIR / "model").mkdir(parents=True, exist_ok=True)
MODEL_FILE = OUTPUT_DIR / "model" / "vision_model.keras"
CLASS_FILE = OUTPUT_DIR / "class_names.json"
INFO_FILE = OUTPUT_DIR / "model" / "model_info.json"

model.save(MODEL_FILE)
CLASS_FILE.write_text(json.dumps(CLASS_NAMES, indent=2))

dataset_commit = None
if (DATA_DIR / ".git").exists():
    dataset_commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=DATA_DIR, capture_output=True, text=True).stdout.strip()

model_info = {
    "class_names": CLASS_NAMES,
    "input_size": [IMAGE_SIZE, IMAGE_SIZE, 3],
    "input_value_range": "0-255 (model rescales internally)",
    "preprocessing": "EXIF transpose, alpha on white, RGB, pad to square with white, bilinear resize",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "preprocessing_fingerprint": PREPROCESSING_FINGERPRINT,
    "test_accuracy": round(float(test_accuracy), 4),
    "test_images": int(len(y_test)),
    "epochs_trained": len(history.history["loss"]),
    "quick_test": QUICK_TEST,
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
    "python_version": sys.version.split()[0],
    "dataset_source": DATASET_SOURCE,
    "dataset_commit": dataset_commit,
    "trained_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
INFO_FILE.write_text(json.dumps(model_info, indent=2))

for f in (MODEL_FILE, CLASS_FILE, INFO_FILE):
    print(f"Saved {f}  ({f.stat().st_size / 1024:.0f} KB)")

## 12. Reload test

Load the saved file into a **new** model object and check that it gives the same predictions as the model in memory. This proves the file is complete and will work in the website.

In [ ]:
reloaded = keras.models.load_model(MODEL_FILE, compile=False)
reloaded_classes = json.loads(CLASS_FILE.read_text())

sample = x_test[:64].astype("float32")
original = model(sample, training=False).numpy()
again = reloaded(sample, training=False).numpy()

assert reloaded_classes == CLASS_NAMES, "class_names.json does not match!"
saved_info = json.loads(INFO_FILE.read_text())
assert saved_info["confidence_threshold"] == CONFIDENCE_THRESHOLD
assert saved_info["preprocessing_fingerprint"], "Preprocessing fingerprint missing!"
assert reloaded.output_shape[-1] == len(CLASS_NAMES)
assert np.allclose(original, again, atol=1e-5), "Reloaded model gives different predictions!"
print("Reload test passed: the saved model gives the same predictions.")
print("Reloaded model accuracy on test set:", accuracy_score(y_test, reloaded.predict(test_ds, verbose=0).argmax(1)))

## 13. Download the files to your computer

The next cell zips `vision_outputs` into **`vision_outputs.zip`** and downloads it.

* If the download does not start, click the **folder icon** in Colab's left sidebar, right-click `vision_outputs.zip` and choose **Download**.
* Colab deletes files when the session ends, so download before you close the tab.

**After downloading**, unzip it and copy into your project folder:
* `vision_outputs/class_names.json` to `project03_vision/class_names.json` (replace the existing one)
* `vision_outputs/model/vision_model.keras` and `model_info.json` to `project03_vision/model/`

Finally, use **File > Download > Download .ipynb** to save this notebook **with its outputs** (graphs, accuracy, confusion matrix) and replace `notebook.ipynb` in your project with it. That is the version to submit.

In [ ]:
zip_path = Path("vision_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUTPUT_DIR.rglob("*"):
        z.write(f, f.relative_to(OUTPUT_DIR.parent))
print(f"Created {zip_path} ({zip_path.stat().st_size / 1024:.0f} KB)")
print(f"TensorFlow version used for training: {tf.__version__}  <- install this version on your computer")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Not running in Colab: the zip file is in", zip_path.resolve())

## 14. Real-photo transfer-learning candidate (keeps the current model intact)

The Fruits 360 images have a controlled white background and do not represent phone photos well. This candidate workflow uses the public **Fruit and Vegetables** Kaggle dataset by youssef salah zakria, pinned to dataset version 5 (CC0; 5.43 GB). It has separate `train`, `validation` and `test` folders and includes Apple, Avocado, Banana, Eggplant and Tomato. Only those five classes are used.

The candidate uses ImageNet-pretrained MobileNetV2 with moderate augmentation, class weighting and fine-tuning. The provider's test split is kept separate and evaluated after training. The candidate is saved under `vision_candidate/`; it never overwrites `vision_outputs/` or the website's current model.

### Run and compare

1. Upload this notebook to Google Colab and select **Runtime > Change runtime type > T4 GPU**. Choose **Runtime > Run all**. The existing baseline notebook workflow runs first; then the candidate section downloads dataset version 5. The download cell will ask you to upload a Kaggle API token (`kaggle.json`). Create it in your Kaggle account settings and upload it only to this private Colab runtime.
2. Review the printed training/validation/test counts, per-class precision/recall, overall test accuracy and confusion matrix.
3. In the final candidate cell, upload the current baseline model from `vision_outputs/model/vision_model.keras` on your computer and the exact red apple photo that was misclassified. The cell prints complete before/after scores and only passes its promotion gate if the photo predicts Apple, test accuracy is at least 90%, and every class recall is at least 85%.
4. Download `vision_candidate.zip` only after reviewing those results. Keep the existing website model in place unless the exact-photo comparison and all five class metrics pass. A failed gate means the candidate is not ready to deploy.

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

CANDIDATE_DATASET_ID = "youssefsalahzakria/fruit-and-vegetables-classification"
CANDIDATE_DATASET_VERSION = "5"
CANDIDATE_DATA_ROOT = Path("/content/real_fruit_data")
CANDIDATE_CLASSES = ["Apple", "Avocado", "Banana", "Eggplant", "Tomato"]
IN_COLAB = "google.colab" in sys.modules


def class_key(name):
    return re.sub(r"[^a-z]", "", name.casefold())


def candidate_split_paths(dataset_root):
    split_names = {"train": {"train", "training"}, "validation": {"valid", "validation"}, "test": {"test"}}
    possible_parents = {dataset_root}
    for path in dataset_root.rglob("*"):
        if path.is_dir() and any(path.name.casefold() in names for names in split_names.values()):
            possible_parents.add(path.parent)

    expected = {class_key(name) for name in CANDIDATE_CLASSES}
    for parent in sorted(possible_parents):
        found = {}
        for split, names in split_names.items():
            split_path = next((parent / name for name in names if (parent / name).is_dir()), None)
            if split_path is None:
                break
            classes = {class_key(folder.name): folder for folder in split_path.iterdir() if folder.is_dir()}
            if not expected.issubset(classes):
                break
            found[split] = {name: classes[class_key(name)] for name in CANDIDATE_CLASSES}
        if len(found) == 3:
            return found
    raise FileNotFoundError("Could not find train, validation and test folders containing all five required class folders.")


candidate_splits = None
if CANDIDATE_DATA_ROOT.exists():
    try:
        candidate_splits = candidate_split_paths(CANDIDATE_DATA_ROOT)
    except FileNotFoundError:
        pass

if candidate_splits is None:
    if IN_COLAB:
        files = __import__("google.colab.files", fromlist=["upload"])
    else:
        raise RuntimeError("Run this candidate-data cell in Google Colab, or set CANDIDATE_DATA_ROOT to an extracted compatible dataset.")

    kaggle_dir = Path.home() / ".kaggle"
    kaggle_json = kaggle_dir / "kaggle.json"
    if not kaggle_json.is_file():
        print("Upload your Kaggle API token file (kaggle.json). It is used only in this Colab runtime.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("Upload the Kaggle token with its original name: kaggle.json")
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        kaggle_json.write_bytes(uploaded["kaggle.json"])
        os.chmod(kaggle_json, 0o600)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    CANDIDATE_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "kaggle", "datasets", "download", "-d", CANDIDATE_DATASET_ID,
        "--dataset-version-number", CANDIDATE_DATASET_VERSION,
        "-p", str(CANDIDATE_DATA_ROOT), "--unzip",
    ], check=True)
    candidate_splits = candidate_split_paths(CANDIDATE_DATA_ROOT)

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
candidate_files = {}
for split, class_dirs in candidate_splits.items():
    candidate_files[split] = {}
    for class_index, class_name in enumerate(CANDIDATE_CLASSES):
        paths = sorted(path for path in class_dirs[class_name].rglob("*") if path.suffix.casefold() in IMAGE_SUFFIXES)
        if not paths:
            raise ValueError(f"No images found for {class_name} in {split} split.")
        candidate_files[split][class_name] = paths
        print(f"{split:10} {class_name:8} {len(paths):5} images")

print("Dataset:", CANDIDATE_DATASET_ID, "version", CANDIDATE_DATASET_VERSION)
print("Split folders verified; no images are moved between the provider's train/validation/test splits.")

In [ ]:
import numpy as np
import tensorflow as tf
import keras
from keras import layers
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

CANDIDATE_IMAGE_SIZE = 224
CANDIDATE_BATCH_SIZE = 32
CANDIDATE_SEED = 42
CANDIDATE_EPOCHS_HEAD = 12
CANDIDATE_EPOCHS_FINE_TUNE = 8
AUTOTUNE = tf.data.AUTOTUNE
keras.utils.set_random_seed(CANDIDATE_SEED)


def preprocess_candidate_image(image, image_size=CANDIDATE_IMAGE_SIZE):
    image = ImageOps.exif_transpose(image)
    has_alpha = image.mode in ("RGBA", "LA", "PA") or (
        image.mode == "P" and "transparency" in image.info
    )
    if has_alpha:
        image = image.convert("RGBA")
        white = Image.new("RGBA", image.size, (255, 255, 255, 255))
        image = Image.alpha_composite(white, image)
    image = image.convert("RGB")
    width, height = image.size
    if width != height:
        side = max(width, height)
        square = Image.new("RGB", (side, side), (255, 255, 255))
        square.paste(image, ((side - width) // 2, (side - height) // 2))
        image = square
    if image.size != (image_size, image_size):
        image = image.resize((image_size, image_size), Image.Resampling.BILINEAR)
    return np.asarray(image, dtype=np.uint8)


def load_candidate_example(path, label):
    def read_image(path_tensor):
        filename = path_tensor.numpy().decode("utf-8")
        with Image.open(filename) as image:
            return preprocess_candidate_image(image)

    pixels = tf.py_function(read_image, [path], Tout=tf.uint8)
    pixels.set_shape((CANDIDATE_IMAGE_SIZE, CANDIDATE_IMAGE_SIZE, 3))
    return tf.cast(pixels, tf.float32), label


def make_candidate_dataset(split, training=False):
    paths = []
    labels = []
    for class_index, class_name in enumerate(CANDIDATE_CLASSES):
        class_paths = candidate_files[split][class_name]
        paths.extend(str(path) for path in class_paths)
        labels.extend([class_index] * len(class_paths))
    dataset = tf.data.Dataset.from_tensor_slices((paths, np.asarray(labels, dtype=np.int32)))
    if training:
        dataset = dataset.shuffle(len(paths), seed=CANDIDATE_SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load_candidate_example, num_parallel_calls=AUTOTUNE)
    return dataset.batch(CANDIDATE_BATCH_SIZE).prefetch(AUTOTUNE)


candidate_train_ds = make_candidate_dataset("train", training=True)
candidate_valid_ds = make_candidate_dataset("validation")
candidate_test_ds = make_candidate_dataset("test")
train_counts = np.asarray([len(candidate_files["train"][name]) for name in CANDIDATE_CLASSES], dtype=np.float32)
class_weights = {index: float(train_counts.sum() / (len(CANDIDATE_CLASSES) * count)) for index, count in enumerate(train_counts)}
print("Training class weights:", dict(zip(CANDIDATE_CLASSES, class_weights.values())))

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08, fill_mode="constant", fill_value=255.0),
    layers.RandomZoom(0.12, fill_mode="constant", fill_value=255.0),
    layers.RandomTranslation(0.06, 0.06, fill_mode="constant", fill_value=255.0),
    layers.RandomContrast(0.15),
], name="real_photo_augmentation")

backbone = keras.applications.MobileNetV2(
    input_shape=(CANDIDATE_IMAGE_SIZE, CANDIDATE_IMAGE_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
backbone.trainable = False
inputs = keras.Input(shape=(CANDIDATE_IMAGE_SIZE, CANDIDATE_IMAGE_SIZE, 3), name="image")
x = augmentation(inputs)
x = layers.Rescaling(1.0 / 127.5, offset=-1, name="mobilenet_normalize")(x)
x = backbone(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(len(CANDIDATE_CLASSES), activation="softmax", name="probabilities")(x)
candidate_model = keras.Model(inputs, outputs, name="real_photo_mobilenetv2")
candidate_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=1),
]
head_history = candidate_model.fit(
    candidate_train_ds, validation_data=candidate_valid_ds,
    epochs=CANDIDATE_EPOCHS_HEAD, class_weight=class_weights, callbacks=callbacks, verbose=2,
)

backbone.trainable = True
for layer in backbone.layers[:-40]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
candidate_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
fine_tune_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7, verbose=1),
]
fine_tune_history = candidate_model.fit(
    candidate_train_ds, validation_data=candidate_valid_ds,
    initial_epoch=len(head_history.history["loss"]),
    epochs=len(head_history.history["loss"]) + CANDIDATE_EPOCHS_FINE_TUNE,
    class_weight=class_weights, callbacks=fine_tune_callbacks, verbose=2,
)


In [ ]:
candidate_test_probabilities = candidate_model.predict(candidate_test_ds, verbose=0)
candidate_true_labels = np.concatenate([labels.numpy() for _, labels in candidate_test_ds])
candidate_predicted_labels = candidate_test_probabilities.argmax(axis=1)
candidate_test_accuracy = accuracy_score(candidate_true_labels, candidate_predicted_labels)
candidate_report = classification_report(
    candidate_true_labels, candidate_predicted_labels,
    labels=np.arange(len(CANDIDATE_CLASSES)), target_names=CANDIDATE_CLASSES,
    output_dict=True, zero_division=0,
)
candidate_matrix = confusion_matrix(
    candidate_true_labels, candidate_predicted_labels,
    labels=np.arange(len(CANDIDATE_CLASSES)),
)
print(f"Real-photo held-out test accuracy: {candidate_test_accuracy:.4f} ({candidate_test_accuracy:.2%})")
print(classification_report(
    candidate_true_labels, candidate_predicted_labels,
    labels=np.arange(len(CANDIDATE_CLASSES)), target_names=CANDIDATE_CLASSES,
    digits=4, zero_division=0,
))
print("Confusion matrix (rows=true, columns=predicted):")
print(candidate_matrix)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(candidate_matrix, display_labels=CANDIDATE_CLASSES).plot(ax=ax, cmap="Greens", colorbar=False)
ax.set_title("Real-photo candidate: held-out test confusion matrix")
plt.tight_layout()
plt.show()

candidate_dir = Path("vision_candidate")
candidate_model_dir = candidate_dir / "model"
candidate_model_dir.mkdir(parents=True, exist_ok=True)
candidate_model_path = candidate_model_dir / "vision_model.keras"
candidate_model.save(candidate_model_path)
(candidate_dir / "class_names.json").write_text(json.dumps(CANDIDATE_CLASSES, indent=2))

if "PREPROCESSING_FINGERPRINT" not in globals():
    raise RuntimeError("Run the notebook's existing preprocessing cell first so the candidate records the app-verified fingerprint.")
candidate_info = {
    "class_names": CANDIDATE_CLASSES,
    "input_size": [CANDIDATE_IMAGE_SIZE, CANDIDATE_IMAGE_SIZE, 3],
    "input_value_range": "0-255 (model rescales internally to -1..1)",
    "preprocessing": "EXIF transpose, alpha on white, RGB, pad to square with white, bilinear resize",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "preprocessing_fingerprint": PREPROCESSING_FINGERPRINT,
    "test_accuracy": round(float(candidate_test_accuracy), 4),
    "test_images": int(len(candidate_true_labels)),
    "per_class_recall": {name: round(float(candidate_report[name]["recall"]), 4) for name in CANDIDATE_CLASSES},
    "quick_test": False,
    "dataset": CANDIDATE_DATASET_ID,
    "dataset_version": CANDIDATE_DATASET_VERSION,
    "model_architecture": "ImageNet MobileNetV2 transfer learning and fine-tuning",
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
    "trained_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(candidate_model_dir / "model_info.json").write_text(json.dumps(candidate_info, indent=2))
print("Saved separate candidate model:", candidate_model_path.resolve())

if IN_COLAB:
    files = __import__("google.colab.files", fromlist=["upload"])
    print("Upload BOTH the current baseline vision_model.keras and the exact red apple JPEG used on the website.")
    uploaded_files = files.upload()
    baseline_paths = [Path(name) for name in uploaded_files if name.casefold().endswith(".keras")]
    apple_paths = [Path(name) for name in uploaded_files if Path(name).suffix.casefold() in IMAGE_SUFFIXES]
    if len(baseline_paths) != 1 or len(apple_paths) != 1:
        raise ValueError("Upload exactly one baseline .keras model and exactly one red apple image in this cell.")

    baseline_model = keras.models.load_model(baseline_paths[0], compile=False)
    with Image.open(apple_paths[0]) as apple_image:
        baseline_size = int(baseline_model.input_shape[1])
        baseline_pixels = preprocess_candidate_image(apple_image, baseline_size).astype(np.float32)[None, ...]
        candidate_pixels = preprocess_candidate_image(apple_image, CANDIDATE_IMAGE_SIZE).astype(np.float32)[None, ...]
    baseline_probabilities = baseline_model(baseline_pixels, training=False).numpy()[0]
    candidate_apple_probabilities = candidate_model(candidate_pixels, training=False).numpy()[0]
    baseline_order = json.loads(Path("vision_outputs/class_names.json").read_text())
    print("Exact red apple image:", apple_paths[0].name)
    print("Baseline scores:", {name: round(float(score), 6) for name, score in zip(baseline_order, baseline_probabilities)})
    print("Candidate scores:", {name: round(float(score), 6) for name, score in zip(CANDIDATE_CLASSES, candidate_apple_probabilities)})
    baseline_prediction = baseline_order[int(baseline_probabilities.argmax())]
    candidate_apple_prediction = CANDIDATE_CLASSES[int(candidate_apple_probabilities.argmax())]
    print("Before:", baseline_prediction, "| After:", candidate_apple_prediction)

    promote_candidate = (
        candidate_apple_prediction == "Apple"
        and candidate_test_accuracy >= 0.90
        and all(candidate_report[name]["recall"] >= 0.85 for name in CANDIDATE_CLASSES)
    )
    print("Promotion gate:", "PASS - candidate may be evaluated for deployment" if promote_candidate else "FAIL - keep the current model; do not promote")
else:
    print("Exact-image A/B comparison requires running this section in Colab and uploading the baseline model plus the original apple photo.")

candidate_zip = Path("vision_candidate.zip")
with zipfile.ZipFile(candidate_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in candidate_dir.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(candidate_dir.parent))
print("Created separate candidate bundle:", candidate_zip.resolve())